In [0]:
from pyspark.sql.functions import regexp_extract, col
from pyspark.sql import functions as F

BRONZE_CAMARA = "/Volumes/workspace/pi_ii_bronze/camara"
BRONZE_SENADO = "/Volumes/workspace/pi_ii_bronze/senado"

def ler_entidade_por_uf(base_path: str, entidade: str, prefixo_uf: str):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("delimiter", ";")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", "true")
        .csv(f"{base_path}/*/{entidade}_*.csv")
        .withColumn("_caminho", col("_metadata.file_path"))
        .withColumn("uf_particao", regexp_extract("_caminho", rf"/{prefixo_uf}/([a-z]{{2}})/", 1))
        .withColumn("ano_particao", regexp_extract("_caminho", rf"{entidade}_(\d{{4}})\.csv", 1))
        .drop("_caminho")
    )
    return df

def ler_cadastro_por_uf(base_path: str, arquivo: str, prefixo_uf: str):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("delimiter", ";")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", "true")
        .csv(f"{base_path}/*/{arquivo}")
        .withColumn("_caminho", col("_metadata.file_path"))
        .withColumn("uf_particao", regexp_extract("_caminho", rf"/{prefixo_uf}/([a-z]{{2}})/", 1))
        .drop("_caminho")
    )
    return df

df_deputados = ler_cadastro_por_uf(BRONZE_CAMARA, "deputados.csv", "camara")
df_despesas_camara = ler_entidade_por_uf(BRONZE_CAMARA, "despesas", "camara")
df_proposicoes = ler_entidade_por_uf(BRONZE_CAMARA, "proposicoes", "camara")
df_proposicoes_autores = ler_entidade_por_uf(BRONZE_CAMARA, "proposicoes_autores", "camara")
df_votacoes_camara = ler_entidade_por_uf(BRONZE_CAMARA, "votacoes", "camara")
df_votos_camara = ler_entidade_por_uf(BRONZE_CAMARA, "votos", "camara")

df_senadores = ler_cadastro_por_uf(BRONZE_SENADO, "senadores.csv", "senado")
df_senadores_exercicios = ler_cadastro_por_uf(BRONZE_SENADO, "senadores_exercicios.csv", "senado")
df_despesas_senado = ler_entidade_por_uf(BRONZE_SENADO, "despesas", "senado")
df_materias = ler_entidade_por_uf(BRONZE_SENADO, "materias", "senado")
df_materias_autorias = ler_entidade_por_uf(BRONZE_SENADO, "materias_autorias", "senado")
df_votacoes_senado = ler_entidade_por_uf(BRONZE_SENADO, "votacoes", "senado")
df_votos_senado = ler_entidade_por_uf(BRONZE_SENADO, "votos", "senado")

In [0]:
df_despesas_camara.select("uf_particao", "ano_particao").distinct().display()

In [0]:
df_despesas_camara.printSchema()
df_proposicoes.printSchema()
df_votos_camara.printSchema()
df_despesas_senado.printSchema()
df_materias.printSchema()

print(df_despesas_camara.count())

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, trim, upper, col, expr

df_proposicoes_silver = (
    df_proposicoes
    .select(
        expr("try_cast(id as int)").alias("id_proposicao"),
        upper(trim(col("siglaTipo"))).alias("sigla_tipo"),
        expr("try_cast(numero as int)").alias("numero"),
        expr("try_cast(ano as int)").alias("ano_proposicao"),
        col("descricaoTipo").alias("descricao_tipo"),
        col("ementa").alias("ementa"),
        expr("try_cast(dataApresentacao as date)").alias("data_apresentacao"),
        expr("try_cast(ultimoStatus_dataHora as timestamp)").alias("ultimo_status_data_hora"),
        col("ultimoStatus_siglaOrgao").alias("ultimo_status_sigla_orgao"),
        col("ultimoStatus_descricaoTramitacao").alias("ultimo_status_descricao_tramitacao"),
        col("ultimoStatus_descricaoSituacao").alias("ultimo_status_descricao_situacao"),
        col("ultimoStatus_apreciacao").alias("ultimo_status_apreciacao"),
        col("urlInteiroTeor").alias("url_inteiro_teor"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_proposicao").isNotNull())
    .dropDuplicates(["id_proposicao"])
)

df_proposicoes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.proposicoes")

In [0]:
total = df_proposicoes.count()
corrompidas = df_proposicoes_silver.filter(
    col("numero").isNull() | col("ano_proposicao").isNull()
).count()

print(f"Total: {total} | Corrompidas: {corrompidas} | {corrompidas/total*100:.4f}%")

import subprocess
resultado = subprocess.run(
    ["grep", "-rl", "74856", "/Volumes/workspace/pi_ii_bronze/camara/"],
    capture_output=True, text=True
)
print(resultado.stdout)

In [0]:
df_votos_camara_silver = (
    df_votos_camara
    .select(
        col("idVotacao").alias("id_votacao"),
        to_timestamp(col("dataHoraVoto")).alias("data_hora_voto"),
        upper(trim(col("voto"))).alias("voto"),
        col("deputado_id").cast("int").alias("id_deputado"),
        trim(col("deputado_nome")).alias("nome_deputado"),
        upper(trim(col("deputado_siglaPartido"))).alias("partido"),
        upper(trim(col("deputado_siglaUf"))).alias("uf_deputado"),
        col("deputado_idLegislatura").cast("int").alias("id_legislatura"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_votacao").isNotNull() & col("id_deputado").isNotNull())
    .dropDuplicates(["id_votacao", "id_deputado"])
)

df_votos_camara_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.votos_camara")

In [0]:
df_despesas_senado_silver = (
    df_despesas_senado
    .select(
        col("id").alias("id_despesa"),
        col("codSenador").cast("int").alias("id_senador"),
        trim(col("nomeSenador")).alias("nome_senador"),
        col("tipoDespesa").alias("tipo_despesa"),
        trim(col("fornecedor")).alias("fornecedor"),
        col("cpfCnpj").alias("cnpj_cpf_fornecedor"),
        col("data").alias("data_despesa"),
        col("ano").alias("ano"),
        col("mes").cast("int").alias("mes"),
        col("valorReembolsado").cast("decimal(12,2)").alias("valor_reembolsado"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_despesa").isNotNull())
    .dropDuplicates(["id_despesa"])
)

df_despesas_senado_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.despesas_senado")

In [0]:

df_despesas_camara_silver = (
    df_despesas_camara
    .select(
        col("ideDocumento").alias("id_documento"),
        col("nuDeputadoId").alias("id_deputado"),
        trim(col("txNomeParlamentar")).alias("nome_parlamentar"),
        upper(trim(col("sgUF"))).alias("uf"),
        upper(trim(col("sgPartido"))).alias("partido"),
        col("numAno").cast("int").alias("ano"),
        col("numMes").cast("int").alias("mes"),
        col("txtDescricao").alias("tipo_despesa"),
        trim(col("txtFornecedor")).alias("fornecedor"),
        col("txtCNPJCPF").alias("cnpj_cpf_fornecedor"),
        to_date(col("datEmissao")).alias("data_emissao"),
        col("vlrDocumento").cast("decimal(12,2)").alias("valor_documento"),
        col("vlrGlosa").cast("decimal(12,2)").alias("valor_glosa"),
        col("vlrLiquido").cast("decimal(12,2)").alias("valor_liquido"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_documento").isNotNull())
    .dropDuplicates(["id_documento"])
)

df_despesas_camara_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.despesas_camara")

In [0]:
df_materias_silver = (
    df_materias
    .select(
        col("Materia_Codigo").cast("int").alias("id_materia"),
        col("Materia_Sigla").alias("sigla"),
        col("Materia_Numero").cast("int").alias("numero"),
        col("Materia_Ano").cast("int").alias("ano_materia"),
        col("Materia_Ementa").alias("ementa"),
        to_date(col("Materia_Data")).alias("data_materia"),
        col("IndicadorAutorPrincipal").alias("indicador_autor_principal"),
        col("IndicadorOutrosAutores").alias("indicador_outros_autores"),
        col("_codigo_parlamentar_consulta").cast("int").alias("id_senador"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("_registro_valido_leg57").alias("registro_valido_leg57"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_materia").isNotNull())
    .dropDuplicates(["id_materia", "id_senador"])
)

df_materias_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.materias")

In [0]:
df_deputados_silver = (
    df_deputados
    .select(
        expr("try_cast(id as int)").alias("id_deputado"),
        trim(col("nome")).alias("nome"),
        upper(trim(col("siglaPartido"))).alias("partido"),
        upper(trim(col("siglaUf"))).alias("uf"),
        col("urlFoto").alias("url_foto"),
        col("uf_particao"),
    )
    .filter(col("id_deputado").isNotNull())
    .dropDuplicates(["id_deputado"])
)
df_deputados_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.deputados")

df_senadores_silver = (
    df_senadores
    .select(
        col("codigo_parlamentar").cast("int").alias("id_senador"),
        trim(col("nome_parlamentar")).alias("nome"),
        upper(trim(col("partido"))).alias("partido"),
        upper(trim(col("uf"))).alias("uf"),
        col("uf_particao"),
    )
    .filter(col("id_senador").isNotNull())
    .dropDuplicates(["id_senador"])
)
df_senadores_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.senadores")

In [0]:

import io
from contextlib import redirect_stdout

dataframes = {
    "deputados": df_deputados,
    "senadores": df_senadores,
    "senadores_exercicios": df_senadores_exercicios,
    "proposicoes_autores": df_proposicoes_autores,
    "materias_autorias": df_materias_autorias,
    "votacoes_camara": df_votacoes_camara,
    "votacoes_senado": df_votacoes_senado,
    "votos_senado": df_votos_senado,
}

buffer = io.StringIO()
with redirect_stdout(buffer):
    for nome, df in dataframes.items():
        print(f"===== {nome} =====")
        df.printSchema()

print(buffer.getvalue())

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, trim, upper, col, expr

df_deputados_silver = (
    df_deputados
    .select(
        expr("try_cast(id as int)").alias("id_deputado"),
        trim(col("nome")).alias("nome"),
        upper(trim(col("siglaPartido"))).alias("partido"),
        upper(trim(col("siglaUf"))).alias("uf"),
        expr("try_cast(idLegislatura as int)").alias("id_legislatura"),
        col("urlFoto").alias("url_foto"),
        col("email").alias("email"),
        col("uf_particao"),
    )
    .filter(col("id_deputado").isNotNull())
    .dropDuplicates(["id_deputado"])
)
df_deputados_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.deputados")

In [0]:
df_senadores_silver = (
    df_senadores
    .select(
        expr("try_cast(codigo_parlamentar as int)").alias("id_senador"),
        trim(col("nome_parlamentar")).alias("nome"),
        upper(trim(col("partido"))).alias("partido"),
        upper(trim(col("uf"))).alias("uf"),
        col("primeiro_exercicio_leg57").alias("primeiro_exercicio_leg57"),
        col("ultimo_fim_exercicio_leg57").alias("ultimo_fim_exercicio_leg57"),
        col("participacoes").alias("participacoes"),
        col("uf_particao"),
    )
    .filter(col("id_senador").isNotNull())
    .dropDuplicates(["id_senador"])
)
df_senadores_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.senadores")

In [0]:
df_senadores_exercicios_silver = (
    df_senadores_exercicios
    .select(
        expr("try_cast(codigo_exercicio as int)").alias("id_exercicio"),
        expr("try_cast(codigo_parlamentar as int)").alias("id_senador"),
        trim(col("nome_parlamentar")).alias("nome"),
        upper(trim(col("partido"))).alias("partido"),
        upper(trim(col("uf"))).alias("uf"),
        expr("try_cast(codigo_mandato as int)").alias("id_mandato"),
        col("participacao").alias("participacao"),
        col("data_inicio_exercicio").alias("data_inicio_exercicio"),
        col("data_fim_exercicio").alias("data_fim_exercicio"),
        col("sigla_causa_afastamento").alias("sigla_causa_afastamento"),
        col("descricao_causa_afastamento").alias("descricao_causa_afastamento"),
        col("uf_particao"),
    )
    .filter(col("id_exercicio").isNotNull())
    .dropDuplicates(["id_exercicio"])
)
df_senadores_exercicios_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.senadores_exercicios")

In [0]:
df_proposicoes_autores_silver = (
    df_proposicoes_autores
    .select(
        expr("try_cast(idProposicao as int)").alias("id_proposicao"),
        expr("try_cast(idDeputadoAutor as int)").alias("id_deputado_autor"),
        col("tipoAutor").alias("tipo_autor"),
        trim(col("nomeAutor")).alias("nome_autor"),
        upper(trim(col("siglaPartidoAutor"))).alias("partido_autor"),
        upper(trim(col("siglaUFAutor"))).alias("uf_autor"),
        expr("try_cast(ordemAssinatura as int)").alias("ordem_assinatura"),
        expr("try_cast(proponente as int)").alias("proponente"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_proposicao").isNotNull() & col("id_deputado_autor").isNotNull())
    .dropDuplicates(["id_proposicao", "id_deputado_autor"])
)
df_proposicoes_autores_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.proposicoes_autores")

In [0]:
df_materias_autorias_silver = (
    df_materias_autorias
    .select(
        expr("try_cast(Materia_Codigo as int)").alias("id_materia"),
        col("Materia_Sigla").alias("sigla"),
        expr("try_cast(Materia_Numero as int)").alias("numero"),
        expr("try_cast(Materia_Ano as int)").alias("ano_materia"),
        expr("try_cast(_codigo_parlamentar_consulta as int)").alias("id_senador"),
        col("IndicadorAutorPrincipal").alias("indicador_autor_principal"),
        col("IndicadorOutrosAutores").alias("indicador_outros_autores"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("_registro_valido_leg57").alias("registro_valido_leg57"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_materia").isNotNull() & col("id_senador").isNotNull())
    .dropDuplicates(["id_materia", "id_senador"])
)
df_materias_autorias_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.materias_autorias")

In [0]:
df_votacoes_camara_silver = (
    df_votacoes_camara
    .select(
        col("id").alias("id_votacao"),
        col("data").alias("data_votacao"),
        col("dataHoraRegistro").alias("data_hora_registro"),
        upper(trim(col("siglaOrgao"))).alias("sigla_orgao"),
        expr("try_cast(aprovacao as int)").alias("aprovacao"),
        expr("try_cast(votosSim as int)").alias("votos_sim"),
        expr("try_cast(votosNao as int)").alias("votos_nao"),
        expr("try_cast(votosOutros as int)").alias("votos_outros"),
        col("descricao").alias("descricao"),
        expr("try_cast(ultimaApresentacaoProposicao_idProposicao as int)").alias("id_proposicao"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_votacao").isNotNull())
    .dropDuplicates(["id_votacao"])
)
df_votacoes_camara_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.votacoes_camara")

In [0]:
df_votacoes_senado_silver = (
    df_votacoes_senado
    .select(
        expr("try_cast(CodigoSessaoVotacao as int)").alias("id_votacao"),
        expr("try_cast(CodigoMateria as int)").alias("id_materia"),
        col("DataSessao").alias("data_votacao"),
        col("HoraInicio").alias("hora_inicio"),
        col("Secreta").alias("secreta"),
        col("DescricaoVotacao").alias("descricao"),
        col("Resultado").alias("resultado"),
        expr("try_cast(TotalVotosSim as int)").alias("votos_sim"),
        expr("try_cast(TotalVotosNao as int)").alias("votos_nao"),
        expr("try_cast(TotalVotosAbstencao as int)").alias("votos_abstencao"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_votacao").isNotNull())
    .dropDuplicates(["id_votacao"])
)
df_votacoes_senado_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.votacoes_senado")

In [0]:
df_votos_senado_silver = (
    df_votos_senado
    .select(
        expr("try_cast(_CodigoSessaoVotacao as int)").alias("id_votacao"),
        expr("try_cast(CodigoParlamentar as int)").alias("id_senador"),
        trim(col("NomeParlamentar")).alias("nome_senador"),
        upper(trim(col("SiglaPartido"))).alias("partido"),
        upper(trim(col("SiglaUF"))).alias("uf_senador"),
        upper(trim(col("Voto"))).alias("voto"),
        col("DescricaoVoto").alias("descricao_voto"),
        col("_DataSessao").alias("data_votacao"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("uf_particao"),
        col("ano_particao"),
    )
    .filter(col("id_votacao").isNotNull() & col("id_senador").isNotNull())
    .dropDuplicates(["id_votacao", "id_senador"])
)
df_votos_senado_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.pi_ii_silver.votos_senado")